# AdventureWorks Sales – Data Cleaning Pipeline

> Reusable pandas-based cleaning function for AdventureWorks sales query outputs (type conversion, missing values, status filtering, and edge-case handling). 
This notebook defines and tests a reusable `clean_sales_dataframe` function for AdventureWorks sales query outputs.
It focuses on:
- Date and numeric type coercion
- Handling missing values
- Optional filtering of cancelled/invalid orders

In [ ]:
# Setup & imports

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


## Helper function: `clean_sales_dataframe`

This section defines a reusable cleaning function for AdventureWorks sales query outputs (dates, numeric columns, and optional status-based filtering).

In [ ]:
def clean_sales_dataframe(
    df,
    date_col="OrderDate",
    numeric_cols=None,
    status_col=None,
    exclude_status=None,
    start_date=None,
    end_date=None
):
    """
    Clean and standardize an AdventureWorks sales query result.

    Steps:
    - Convert `date_col` to datetime, dropping rows where conversion fails.
    - Optionally filter to a date range [`start_date`, `end_date`] (inclusive).
    - Coerce specified numeric columns (e.g., SubTotal, LineTotal, OrderQty)
      to numeric, setting non-parsable values to NaN.
    - Optionally drop rows where all specified numeric columns are missing
      after coercion.
    - Optionally filter out rows whose `status_col` is in `exclude_status`
      (e.g., Status values 6 = Cancelled, or other custom invalid codes from
      the source query). [web:6][web:15]

    Edge cases / assumptions:
    - If `date_col` is missing or cannot be converted for a row, that row is removed.
    - If a numeric column in `numeric_cols` is missing from `df`, it is silently ignored.
    - If `status_col` is provided but not present in `df`, no status-based filtering is applied.
    - `exclude_status` can be a single value or an iterable of values; if None,
      no status filtering is done.
    - Status semantics follow AdventureWorks SalesOrderHeader:
      1 = In process; 2 = Approved; 3 = Backordered; 4 = Rejected;
      5 = Shipped; 6 = Cancelled. [web:6][web:52]
    """
    import pandas as pd

    df = df.copy()

    # 1) Date conversion
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        df = df.dropna(subset=[date_col])
    else:
        raise KeyError(f"{date_col} not found in DataFrame columns.")

    # 2) Date range filter
    if start_date is not None:
        df = df[df[date_col] >= pd.to_datetime(start_date)]
    if end_date is not None:
        df = df[df[date_col] <= pd.to_datetime(end_date)]

    # 3) Numeric coercion
    if numeric_cols is None:
        numeric_cols = ["SubTotal", "LineTotal", "OrderQty"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")  # invalid -> NaN [web:12][web:56]

    # Drop rows where all numeric columns are NaN (if any numeric columns exist)
    present_numeric = [c for c in numeric_cols if c in df.columns]
    if present_numeric:
        df = df.dropna(subset=present_numeric, how="all")

    # 4) Status filter (cancelled/invalid)
    if status_col is not None and status_col in df.columns and exclude_status is not None:
        if not isinstance(exclude_status, (list, set, tuple)):
            exclude_values = {exclude_status}
        else:
        # Normalize exclude_status to a set
            exclude_values = set(exclude_status)

        df = df[~df[status_col].isin(exclude_values)]

    # 5) Final tidy-up
    df = df.reset_index(drop=True)
    return df